# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


In [1]:

import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

## Raven test run

In [2]:
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = "CypressRiverT_ERA5"

In [3]:
# raven_name = input("Enter the Raven model name (if different):")
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' and test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_name, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

Exiting without running Raven.exe.


## Calibration set-up



In [4]:
calibrator = spot_setup_rvn(raven_name, DATA_CAL, obj_fn= "nse", split_areas = False, spring_eval= False)

# calibrator.dep_fact = Uniform(low=0.80, high=1.20)
# calibrator.seep_k = Uniform(low=0.00, high=0.10)
# calibrator.basin_fact = Uniform(low=0.80, high=1.20)

# sim_results = calibrator.simulation([1,0.05,1])
# obs_results = calibrator.evaluation()

C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:298: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from ../Data/Calibration/cypress/output\CypressRiverT_ERA5_Hydrographs.csv


In [5]:
# Use the Pareto Archived Dynamically Dimensioned Search (PADDS) algorithm for calibration
# sampler=sp.algorithms.padds(spot_setup=calibrator, # Core model setup object
#                             dbname = "padds_raven", # Save results of cal run to this database
#                             dbformat='csv')

 
sampler=sp.algorithms.rope(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
                           parallel='seq',
                           dbname='rope_rv_nse_nosplit',
                           dbformat='csv')

# sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
#                            parallel='seq',
#                            dbname='mc_raven',
#                            dbformat='csv')



Generated parameters for SpotPy calibration: [(0.9941784 , 'dep_fact', 0.04105, 1.     , 8.000e-01, 1.2, False)
 (0.04264471, 'seep_k', 0.01006, 0.05002, 2.257e-07, 0.1, False)
 (0.87576031, 'basin_fact', 0.03951, 1.     , 8.000e-01, 1.2, False)]


C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:298: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from ../Data/Calibration/cypress/output\CypressRiverT_ERA5_Hydrographs.csv


In [6]:
rep = 30
sampler.sample(rep)

## REMINDER - try to reduce the evaluation period to 2003 onward to improve performance

# setup = calibrator  # your spot_setup object

# params = [p.optguess for p in setup.parameters]
# print("params:", params)

# sim = setup.simulation(params)
# evals = setup.evaluation()
# obj = setup.objectivefunction(sim, evals)
# print("Objective output:", obj)

Starting the ROPE algotrithm with 30 repetitions...
Initializing the  RObust Parameter Estimation (ROPE) algorithm  with  30  repetitions
The objective function will be maximized
SPOTPY will automatically adjust the number of total repetitions.
Generated parameters for SpotPy calibration: [(0.90754956, 'dep_fact', 0.04105, 1.     , 8.000e-01, 1.2, False)
 (0.08070033, 'seep_k', 0.01006, 0.05002, 2.257e-07, 0.1, False)
 (0.99009233, 'basin_fact', 0.03951, 1.     , 8.000e-01, 1.2, False)]
Running: Raven.exe CypressRiverT_ERA5 -o ./output/
STDOUT:
                        RAVEN                               
 a robust semi-distributed hydrological modelling framework 
    Copyright 2008-2024, the Raven Development Team 
                    Version 3.8.0 w/ netCDF
                BuildDate Jan 10 2024
Generating Master Parameter List...
Autocalculating Model Parameters...
...done Autocalculating.
Checking for Required Model Parameters...
...Done Checking
...model input successfully parsed



C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:298: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from ../Data/Calibration/cypress/output\CypressRiverT_ERA5_Hydrographs.csv
Simulation: 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96380    0.0
96381    0.0
96382    0.0
96383    0.0
96384    0.0
Name: sub875 [m3/s], Length: 96385, dtype: float64
Evaluation: 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
96380   NaN
96381   NaN
96382   NaN
96383   NaN
96384   NaN
Name: sub875 (observed) [m3/s], Length: 96385, dtype: float64
Simulation type: <class 'pandas.core.series.Series'>
Evaluation type: <class 'pandas.core.series.Series'>
Nash-Sutcliffe Efficiency (NSE): 0.13318160472825602
1 of 30, maximal objective function=0.133182, time remaining: 00:04:23
Initialize database...
['csv', 'hdf5', 'ram', 'sql', 'custom', 'noData']
* Database file 'rope_rv_nse_nosplit.csv' created.
1 Subset: Run 0 of 15 (best like=0.133182)
Running: Raven.exe CypressRiverT_ERA5 -o ./output/
STDOUT:
                        RAVEN     

C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:298: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from ../Data/Calibration/cypress/output\CypressRiverT_ERA5_Hydrographs.csv
Simulation: 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96380    0.0
96381    0.0
96382    0.0
96383    0.0
96384    0.0
Name: sub875 [m3/s], Length: 96385, dtype: float64
Evaluation: 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
96380   NaN
96381   NaN
96382   NaN
96383   NaN
96384   NaN
Name: sub875 (observed) [m3/s], Length: 96385, dtype: float64
Simulation type: <class 'pandas.core.series.Series'>
Evaluation type: <class 'pandas.core.series.Series'>
Nash-Sutcliffe Efficiency (NSE): 0.13552989075102395
2 of 30, maximal objective function=0.13553, time remaining: 00:06:07
1 Subset: Run 1 of 15 (best like=0.13553)
Running: Raven.exe CypressRiverT_ERA5 -o ./output/
STDOUT:
                        RAVEN                               
 a robust semi-distributed hydrological modelling framework 
    Copyright 2008-2024, the Raven Dev

...

C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:298: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from ../Data/Calibration/cypress/output\CypressRiverT_ERA5_Hydrographs.csv
Simulation: 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96380    0.0
96381    0.0
96382    0.0
96383    0.0
96384    0.0
Name: sub875 [m3/s], Length: 96385, dtype: float64
Evaluation: 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
96380   NaN
96381   NaN
96382   NaN
96383   NaN
96384   NaN
Name: sub875 (observed) [m3/s], Length: 96385, dtype: float64
Simulation type: <class 'pandas.core.series.Series'>
Evaluation type: <class 'pandas.core.series.Series'>
Nash-Sutcliffe Efficiency (NSE): 0.170637321663771
20 of 30, maximal objective function=0.172099, time remaining: 00:02:42
3 Subset: Run 1 of 3 (best like=0.172099)
Running: Raven.exe CypressRiverT_ERA5 -o ./output/
STDOUT:
                        RAVEN                               
 a robust semi-distributed hydrological modelling framework 
    Copyright 2008-2024, the Raven Dev

C:\Users\tsaleh\Code_Workspace\GIS-Projects\PWA\src\py\raven_py.py:298: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_output = pd.read_csv(output_csv)


Loaded hydrograph data from ../Data/Calibration/cypress/output\CypressRiverT_ERA5_Hydrographs.csv
Simulation: 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
96380    0.0
96381    0.0
96382    0.0
96383    0.0
96384    0.0
Name: sub875 [m3/s], Length: 96385, dtype: float64
Evaluation: 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
96380   NaN
96381   NaN
96382   NaN
96383   NaN
96384   NaN
Name: sub875 (observed) [m3/s], Length: 96385, dtype: float64
Simulation type: <class 'pandas.core.series.Series'>
Evaluation type: <class 'pandas.core.series.Series'>
Nash-Sutcliffe Efficiency (NSE): 0.170637321663771
21 of 30, maximal objective function=0.172099, time remaining: 00:02:24
3 Subset: Run 2 of 3 (best like=0.172099)


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
results_df = sp.analyser.load_csv_results(sampler.dbname)

In [ ]:
results_df = pd.DataFrame(results_df)

In [ ]:
param_nse_sets = results_df[results_df.columns[0:4]]
np.corrcoef(param_nse_sets.transpose())

array([[ 1.        ,  0.26223791, -0.49272106, -0.64936952],
       [ 0.26223791,  1.        , -0.63361712, -0.29272321],
       [-0.49272106, -0.63361712,  1.        ,  0.25272791],
       [-0.64936952, -0.29272321,  0.25272791,  1.        ]])

In [ ]:
from sklearn.metrics import mutual_info_score

param_nse_sets_norm = param_nse_sets.copy()
param_nse_sets_norm.iloc[:, 0] = ((param_nse_sets_norm.iloc[:, 0] - param_nse_sets_norm.iloc[:, 0].min()) \
                                  / (param_nse_sets_norm.iloc[:, 0].max() - param_nse_sets_norm.iloc[:, 0].min())).round(1)
param_nse_sets_norm.iloc[:, 1] = ((param_nse_sets_norm.iloc[:, 1] - 0.8) / 0.4).round(1)
param_nse_sets_norm.iloc[:, 2] = (param_nse_sets_norm.iloc[:, 2]/0.1).round(1)
param_nse_sets_norm.iloc[:, 3] = ((param_nse_sets_norm.iloc[:, 3] - 0.8) / 0.4).round(1)

mi_dep = mutual_info_score(param_nse_sets_norm.iloc[:, 0], param_nse_sets_norm.iloc[:, 1])
mi_seep = mutual_info_score(param_nse_sets_norm.iloc[:, 0], param_nse_sets_norm.iloc[:, 2])
mi_area = mutual_info_score(param_nse_sets_norm.iloc[:, 0], param_nse_sets_norm.iloc[:, 3])

print(f"Mutual Information between dep_fact and nse: {mi_dep}")
print(f"Mutual Information between seep_k and nse: {mi_seep}")
print(f"Mutual Information between basin_area and nse: {mi_area}")

Mutual Information between dep_fact and nse: 0.996797330034217
Mutual Information between seep_k and nse: 1.0345985085253309
Mutual Information between basin_area and nse: 1.2857699770934605


c:\Users\tsaleh\.conda\envs\pwa_dev\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
c:\Users\tsaleh\.conda\envs\pwa_dev\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
c:\Users\tsaleh\.conda\envs\pwa_dev\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:59: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)


In [ ]:
param_nse_sets_norm

,like1,pardep_fact,parseep_k,parbasin_fact
0,0.3,0.5,0.4,0.5
1,0.2,0.5,0.1,0.7
2,0.2,0.7,0.5,0.4
3,1.0,0.8,0.0,0.0
4,0.1,0.5,0.9,0.9
5,0.3,0.2,0.5,0.5
6,0.1,0.3,0.8,0.8
7,0.2,0.8,0.2,0.7
8,0.0,0.7,0.3,0.9
9,0.3,0.0,0.7,0.5


## Additional workflows

- Extract Raven results

- Specify calibration parameters, method, iterations, etc.
- Run calibration
- Store results & optimal parameters

## Update Raven files

In [ ]:
Cypress_rvp = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvp")
Cypress_rvp.rvp_update("base", np.array([0.89955,0.09,0.818696]))

In [ ]:
Cypress_rvh = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvh")

Cypress_rvh.rvh_update("base", np.array([0.89955,0.09,0.818696]))

In [ ]:
rv.run_raven(raven_name, DATA_CAL)

Manual Raven results with same parameters yield a different NSE

In [ ]:
simulation = pd.read_csv(os.path.join(DATA_CAL, "output/"+raven_name+"_Hydrographs.csv"))["sub875 [m3/s]"]

evaluation = pd.read_csv(os.path.join(DATA_CAL, "output/"+raven_name+"_Hydrographs.csv"))["sub875 (observed) [m3/s]"]

nse = sp.objectivefunctions.nashsutcliffe(simulation, evaluation)

nse = np.squeeze(nse).item()

print(f"Nash-Sutcliffe Efficiency (NSE): {nse}")

custom_nse = 1- (np.sum((simulation - evaluation)**2) / np.sum((evaluation - np.mean(evaluation))**2))
print(f"Custom NSE: {custom_nse}")


In [ ]:
kge = sp.objectivefunctions.kge(simulation, evaluation)

print(kge)